# VEGFR2 Activity Prediction - Colab Notebook

This notebook provides a complete workflow for:
1. **Environment setup** - Install dependencies
2. **Data download** - Fetch ChEMBL VEGFR2 IC50 data
3. **Model training** - Train ML (RF, SVM, XGBoost) and GNN (GCN, GAT, MPNN) models
4. **Screening/Inference** - Predict on new compound libraries

## Requirements
- GPU runtime (required for GNN training)
- ~10-15 min for full training

**Enable GPU**: Runtime → Change runtime type → GPU

## 1. Environment Setup

In [ ]:
# Check GPU availability
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU detected. GNN training will be VERY slow on CPU.")
    print("   Enable GPU: Runtime → Change runtime type → GPU")

In [ ]:
# Install dependencies
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q numpy pandas pyyaml scikit-learn xgboost rdkit-pypi optuna

# Verify imports
import torch, numpy, pandas, sklearn, xgboost, rdkit, yaml, optuna
print("✅ All packages installed")

In [ ]:
# Clone the repository
import os
REPO_URL = "https://github.com/YOUR_USERNAME/ai-code.git"  # CHANGE THIS
REPO_DIR = "/content/ai-code"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repository already exists, pulling latest...")
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print(f"Working in: {os.getcwd()}")

In [ ]:
# Install the package in development mode
!pip install -e . -q

# Verify import works
from vegfr2.features import mol_to_graph, ATOM_FEAT_DIM, BOND_FEAT_DIM
from vegfr2.gnn_models import build_model
from vegfr2.ml_models import train_ml_model
print(f"✅ Package imported successfully")
print(f"   Atom feat dim: {ATOM_FEAT_DIM}")
print(f"   Bond feat dim: {BOND_FEAT_DIM}")

## 2. Download Training Data

In [ ]:
# Download ChEMBL VEGFR2 (CHEMBL279) IC50 data
!python scripts/download_data.py --out data/raw/chembl_vegfr2.csv

# Verify data
import pandas as pd
df = pd.read_csv("data/raw/chembl_vegfr2.csv")
print(f"Downloaded {len(df)} compounds")
print(df.head())
print(f"IC50 range: {df['ic50_nM'].min():.1f} - {df['ic50_nM'].max():.1f} nM")

## 3. Train Models

Choose which models to train. Each cell trains one model type.

In [ ]:
# Train Classical ML models (RF, SVM, XGBoost)
# Fast - runs on CPU, ~1-2 minutes
!python scripts/train.py --model rf --config configs/config.yaml
!python scripts/train.py --model svm --config configs/config.yaml
!python scripts/train.py --model xgb --config configs/config.yaml

# Results saved to runs/<model>/model.pkl
# Metrics saved to runs/results.json

In [ ]:
# Train GNN models (GCN, GAT, MPNN) - REQUIRES GPU
# ~5-10 minutes per model on GPU
!python scripts/train.py --model gcn --config configs/config.yaml
!python scripts/train.py --model gat --config configs/config.yaml
!python scripts/train.py --model mpnn --config configs/config.yaml

# With hyperparameter optimization (optional, slower):
# !python scripts/train.py --model gcn --hpo --config configs/config.yaml

# Results saved to runs/<model>/best.pt
# Metrics saved to runs/results.json

In [ ]:
# OR train all models at once
!python scripts/train.py --model all --config configs/config.yaml

# View results summary
import json
with open("runs/results.json") as f:
    results = json.load(f)

print(f"{'Model':<8} {'ACC':>6} {'SEN':>6} {'SPE':>6} {'MCC':>6} {'AUC':>6}")
print("-" * 44)
for name, m in results.items():
    auc_str = f"{m['auc']:.4f}" if m["auc"] is not None else "N/A"
    print(f"{name:<8} {m['acc']:.4f} {m['sen']:.4f} {m['spe']:.4f} {m['mcc']:.4f} {auc_str:>6}")

## 4. Screen New Compound Libraries

Use trained models to predict VEGFR2 activity on new SMILES.

In [ ]:
# Prepare a screening library (example: create test CSV)
import pandas as pd

# Example: Your compound library
library_smiles = [
    "CC(=O)OC1=CC=CC=C1C(=O)O",  # Aspirin
    "CCO",                       # Ethanol
    "C1=CC=CC=C1",               # Benzene
    "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O",  # Ibuprofen
    "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",   # Caffeine
    "CC(C)CC1=CC(=CC=C1)O",    # Naproxen-like
    "C[C@H](O)CC1=CC=CC=C1",   # Chiral molecule (R)
    "C[C@@H](O)CC1=CC=CC=C1",  # Chiral molecule (S)
    "C/C=C/C",                 # E-alkene
    "C/C=C\\C",                # Z-alkene
]

library_df = pd.DataFrame({"smiles": library_smiles})
library_df.to_csv("data/screen_library.csv", index=False)
print(f"Created library with {len(library_df)} compounds")
print(library_df)

In [ ]:
# Screen with a trained GNN model
# Change model_path to your best model
MODEL_PATH = "runs/gcn/best.pt"  # or gat/best.pt, mpnn/best.pt
INPUT_CSV = "data/screen_library.csv"
OUTPUT_CSV = "results/screen_gcn_results.csv"
THRESHOLD = 0.5

!python scripts/screen.py \
    --model {MODEL_PATH} \
    --input {INPUT_CSV} \
    --output {OUTPUT_CSV} \
    --threshold {THRESHOLD} \
    --batch-size 32

# View results
results_df = pd.read_csv(OUTPUT_CSV)
print(results_df[['smiles', 'probability', 'hit']].to_string(index=False))

In [ ]:
# Screen with a trained ML model
MODEL_PATH = "runs/xgb/model.pkl"  # or rf/model.pkl, svm/model.pkl
INPUT_CSV = "data/screen_library.csv"
OUTPUT_CSV = "results/screen_xgb_results.csv"
THRESHOLD = 0.5

!python scripts/screen.py \
    --model {MODEL_PATH} \
    --input {INPUT_CSV} \
    --output {OUTPUT_CSV} \
    --threshold {THRESHOLD} \
    --batch-size 256

# View results
results_df = pd.read_csv(OUTPUT_CSV)
print(results_df[['smiles', 'probability', 'hit']].to_string(index=False))

## 5. Advanced: Custom Training & Screening

In [ ]:
# Custom training with your own config
import yaml

custom_config = {
    "seed": 42,
    "paths": {
        "raw_csv": "data/raw/chembl_vegfr2.csv",
        "output_dir": "runs_custom"
    },
    "label": {"threshold_nM": 500},
    "split": {"test_size": 0.1, "val_frac_of_remaining": 0.111111},
    "fingerprint": {"radius": 2, "n_bits": 2048},
    "gnn": {
        "hidden": 128,
        "layers": 4,
        "heads": 8,
        "batch": 64,
        "lr": 0.0005,
        "epochs": 300,
        "patience": 20
    },
    "hpo": {"n_trials": 30}
}

with open("configs/custom_config.yaml", "w") as f:
    yaml.dump(custom_config, f)

print("Custom config saved. Train with:")
print("!python scripts/train.py --model gcn --config configs/custom_config.yaml")

In [ ]:
# Custom screening with your own library CSV
# Your CSV must have a 'smiles' column

# Example: Load your library
# your_library = pd.read_csv("/content/drive/MyDrive/my_compounds.csv")
# your_library.to_csv("data/my_library.csv", index=False)

# Screen with multiple models and compare
import pandas as pd
import numpy as np

models_to_screen = [
    ("runs/gcn/best.pt", "GCN"),
    ("runs/gat/best.pt", "GAT"),
    ("runs/mpnn/best.pt", "MPNN"),
    ("runs/xgb/model.pkl", "XGBoost"),
    ("runs/rf/model.pkl", "RandomForest"),
]

library_df = pd.read_csv("data/screen_library.csv")
all_results = library_df[['smiles']].copy()

for model_path, model_name in models_to_screen:
    try:
        if model_path.endswith('.pt'):
            !python scripts/screen.py --model {model_path} --input data/screen_library.csv --output results/temp_{model_name}.csv --threshold 0.5 --batch-size 32
        else:
            !python scripts/screen.py --model {model_path} --input data/screen_library.csv --output results/temp_{model_name}.csv --threshold 0.5 --batch-size 256

        res = pd.read_csv(f"results/temp_{model_name}.csv")
        all_results[f"prob_{model_name}"] = res["probability"]
        all_results[f"hit_{model_name}"] = res["hit"]
        print(f"\u2705 {model_name} done")
    except Exception as e:
        print(f"\u274c {model_name} failed: {e}")

# Save combined results
all_results.to_csv("results/combined_screening.csv", index=False)
print("\nCombined results:")
print(all_results.to_string(index=False))

In [ ]:
# Ensemble prediction: average probabilities across models
ensemble_cols = [c for c in all_results.columns if c.startswith('prob_')]
all_results['prob_ensemble'] = all_results[ensemble_cols].mean(axis=1)
all_results['hit_ensemble'] = all_results['prob_ensemble'] >= 0.5

# Sort by ensemble probability
all_results = all_results.sort_values('prob_ensemble', ascending=False)

print("Top predictions (ensemble):")
display_cols = ['smiles', 'prob_ensemble', 'hit_ensemble'] + ensemble_cols
print(all_results[display_cols].to_string(index=False))

## 6. Download Results

In [ ]:
# Download results to local machine
from google.colab import files

# Download combined screening results
files.download("results/combined_screening.csv")

# Download trained models (optional)
# files.download("runs/gcn/best.pt")
# files.download("runs/xgb/model.pkl")

## Tips & Troubleshooting

### GPU Issues
- **Runtime disconnected**: Colab free tier has limits. Use shorter epochs or smaller models.
- **OOM (Out of Memory)**: Reduce `batch` size in config.yaml (e.g., 64 → 32)
- **Slow training**: Ensure GPU is enabled (Runtime → Change runtime type → GPU)

### Data Issues
- **Download fails**: ChEMBL API may be slow. Retry or use a local fallback CSV.
- **Invalid SMILES**: Screen script skips invalid SMILES (shows NaN probability)

### Model Selection
- **Fast screening**: Use ML models (RF, XGBoost) - CPU only, very fast
- **Best accuracy**: GNN models (especially MPNN) - need GPU, capture 3D/steric effects
- **Chiral/E-Z sensitivity**: GNN models now use stereochemistry features (new in v2)

### Custom Library Format
Your screening CSV must have:
```csv
smiles,optional_id,optional_name
"CCO",CMPD001,Ethanol
"CC(=O)OC1=CC=CC=C1C(=O)O",CMPD002,Aspirin
```

### Threshold Tuning
- Default threshold: 0.5 (probability ≥ 0.5 = hit)
- For stricter hits: use 0.7-0.9
- For recall-focused: use 0.3-0.4
- Check `results.json` for model AUC to gauge reliability